# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

J. Smith  ->  P. J. Smith  |  ['J. Smith']
F. Walter  ->  F. Walter  |  ['F. Walter']
Y.-S. Ting  ->  Y.-S. Ting  |  ['Y.-S. Ting']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
N. Neumayer  ->  N. Neumayer  |  ['N. Neumayer']
R. Tress  ->  R. Tress  |  ['R. Tress']
F. Xu  ->  F. Xu  |  ['F. Xu']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']


X. Lin  ->  X. Lin  |  ['X. Lin']
J. Matharu  ->  J. Matharu  |  ['J. Matharu']
Arxiv has 80 new papers today
          9 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/9 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.23367


extracting tarball to tmp_2611.23367...

 done.


J. Smith  ->  P. J. Smith  |  ['J. Smith']
F. Walter  ->  F. Walter  |  ['F. Walter']


Retrieving document from  https://arxiv.org/e-print/2611.23492


... (8 outputs left out)

 done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.24630/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.24630/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


J. Matharu  ->  J. Matharu  |  ['J. Matharu']


Found 206 bibliographic references in tmp_2611.24630/main.bbl.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.23367-b31b1b.svg)](https://arxiv.org/abs/2611.23367) | **JWST Observations of Starbursts: Polycyclic Aromatic Hydrocarbons in the NGC 253 Superwind**  |
|| S. A. Cronin, et al. -- incl., <mark>J. Smith</mark>, <mark>F. Walter</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *18 pages, 5 main figures, 4 appendix figures, submitted to ApJL*|
|**Abstract**|            We present the first spatially resolved study of polycyclic aromatic hydrocarbon (PAH) evolution in the galactic superwind of the nearby (d~3.5Mpc) galaxy NGC 253. Using JWST MIRI and NIRCam, we image ~$\pm$2kpc along the galaxy minor axis, capturing both sides of the starburst-driven wind at ~0.05''-0.375'' (~0.85-6.4pc) resolution and providing one of the most detailed views of the wind to date. We target PAH emission at 3.3um, 7.7um, and 11.3um, revealing PAH-bright filamentary streamers along the limb-brightened edges of the outflow cones out to at least a deprojected distance r~0.5kpc, beyond which they likely become confused with the spiral arms. In the southwest (SW) and northeast (NE) streamers, PAH intensity declines faster than $r^{-2}$, suggesting either PAH destruction or a decrease in gas column density or dust-to-gas ratio rather than a response to declining illumination alone. Band ratios trace changes in PAH properties, with the 11.3/7.7um ratio increasing with distance in the SW and northwest (NW) streamers and 3.3/7.7um and 3.3/11.3um decreasing in the SW and NE streamers, indicating larger, more neutral PAHs farther from the starburst. Given the thermal and dynamical properties of the gas in the SW streamer, cool gas is expected to survive acceleration in the outflow and enable PAH shielding. Compared with those of the prototypical starburst M82, the NGC 253 band ratios indicate faster evolution in PAH size. Thus, the NGC 253 wind may represent an earlier stage of outflow evolution in which small PAHs are destroyed near the starburst within the first few Myr.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.23492-b31b1b.svg)](https://arxiv.org/abs/2611.23492) | **JEPA for astronomy: multi-survey spectral alignment puts stellar parameters on the leading principal axes**  |
|| X. Zhao, R. F. Wyse, <mark>Y.-S. Ting</mark>, A. S. Szalay |
|*Appeared on*| *2026-11-19*|
|*Comments*| *15 pages, 5 figures, 2 tables. Submitted to AAS journals. Comments welcome*|
|**Abstract**|            Spectra from different surveys carry different systematics and are not calibrated against one another. Multi-survey foundation models for stellar spectra are then built by aligning embeddings of the same star observed across surveys. Most in astronomy follow the recipes established for language and vision, aligning with a contrastive loss that separates every star from the others. However, this does not ensure that the stellar physics is encoded in a form that a simple read-out can recover. Holding the cross-survey architecture fixed and varying only the alignment loss, we measure what an objective encodes from the principal-component geometry of the Gaia BP/RP (XP) spectra and from a linear probe of the stellar parameters. In a properly aligned embedding, the leading two principal components form a plane that recovers an analog of the Kiel diagram, the third component tracks [Fe/H], and [alpha/Fe] varies coherently across the plane, such that the chemically defined Galactic disks and accreted populations occupy distinct loci. We show that legibility, the ease of reading the information out, is highest for a training objective that never pushes stars apart, and instead constrains only the overall distribution of each survey's embeddings (JEPA). JEPA also exceeds the contrastive alternative at the same architecture, and the performance improves further as more surveys (Gaia XP+LAMOST LRS+DESI+APOGEE) are included. Predicted abundances for 8.0 million XP giants reproduce the main [alpha/Fe]-[Fe/H] structure of the Milky Way. These results indicate that the choice of alignment objective, not alignment alone, determines whether stellar labels are linearly recoverable.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.24630-b31b1b.svg)](https://arxiv.org/abs/2611.24630) | **The Gas-Phase Metallicity and Nitrogen abundances in typical galaxies at $z\sim5$**  |
|| C. D. Cesare, et al. -- incl., <mark>J. Matharu</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *Main text 16 pages, 8 figures. Main results in Fig. 5 and Fig. 6. Submitted to A&A. Comments are welcome*|
|**Abstract**|            The relation between stellar mass and gas-phase metallicity (MZR) is a key tracer of the baryon cycle in galaxies. JWST now gives access to rest-optical emission lines at z > 3, enabling unprecedented constraints on the chemical enrichment and the conditions of the interstellar medium of galaxies in the early Universe. In this work, we present the MZR at $z=3.8-6.6$ from stacked NIRCam/grism spectra of 1748 H$\alpha$-selected star-forming galaxies compiled over an area of 320 arcmin$^2$. The simple H$\alpha$ flux-limited selection allows us to measure representative [NII]$\lambda$6584/H$\alpha$ and [SII]$\lambda\lambda$6716,6731/ H$\alpha$ ratios in stacks spanning $\rm log(M_\star/M_\odot)=8-10.2$, extending the upper mass range of previous high-redshift MZR measurements by $\sim1$ dex. Gas-phase metallicities are derived from these ratios using strong-line calibrations recently derived from high-redshift galaxies. The[NII]-based MZR has a slope $\gamma = 0.42^{+0.10}_{-0.07}$ and $12+\rm log(O/H) = 7.94^{+0.06}_{-0.09}$ at $\rm log(M_\star/M_\odot) = 9$, in broad agreement with recent empirical studies and hydrodynamical simulations. The [SII]-based relation yields a consistent slope and a $\sim0.3$ dex higher normalization, within the systematic uncertainties of strong-line calibrations. From the resolved [SII] doublet we infer a median electron density of $n_e = 184^{+226}_{-169}\,\rm cm^{-3}$. Combining [NII] and [SII], we indirectly estimate the N/O ratio and find that it follows the trend expected from primary and secondary nitrogen production, with no evidence for nitrogen enhancement in the HII region emission that dominates our galaxies. Our results provide a benchmark for the chemical enrichment and ISM conditions of typical star-forming galaxies at $z\sim5$ across a wide mass range.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.24110-b31b1b.svg)](https://arxiv.org/abs/2611.24110) | **Multi-epoch imaging observations of the Galactic Center with NIRCam and MIRI**  |
|| R. Schödel, et al. -- incl., <mark>N. Neumayer</mark>, <mark>R. Tress</mark>, <mark>F. Xu</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *Submitted as White Paper to STScI*|
|**Abstract**|            The center of the Milky Way is the densest and most extreme environment in our Galaxy, its most active star forming region, and the only galaxy nucleus where we can resolve stars down to about a solar mass on the main sequence. The Galactic Center (GC) contains a nuclear stellar disk, a nuclear star cluster, and a supermassive black hole, all embedded within the so-called Central Molecular Zone, which showcases conditions of the interstellar medium that are typically found in high-redshift star-forming regions. The GC hosts the highest concentration of massive stars in the Milky Way and hosts a large concentration of rare objects, such as Wolf Rayet stars and X-ray binaries. The GC is therefore an astrophysical target of outstanding importance. A multi-epoch JWST program enables novel science including the lifetime of planet-forming disks, the search for intermediate mass black holes, direct measurements of the multi-scale gravitational potential, and a deep measurement of the star formation history. Multi-epoch imaging observations with JWST's NIRCam and MIRI instruments will go far beyond Roman WFI surveys in terms of depth and astrometric and photometric precision because of its $3\times$ better resolution, and it expands spectral coverage in narrow bands and mid-infrared wavelength coverage. The proposed observations would correspond to a Treasury Program of about 420\,h, distributed over three epochs spaced three to five years apart.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: '69117' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.23988-b31b1b.svg)](https://arxiv.org/abs/2611.23988) | **Two Faint Tidal Disruption Events Discovered in a Pilot Search with the Wide Field Survey Telescope**  |
|| M. Wu, et al. -- incl., <mark>Y. Wang</mark>, <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *Accepted for publication in ApJ*|
|**Abstract**|            We report the discovery of TDE 2025alrm/WFST250129blnve and TDE 2025amkg/WFST251021dbmwk in the first six months of the wide-field survey (WFS) performed by the 2.5-meter Wide Field Survey Telescope (WFST). Both events are apparently faint among known optical tidal disruption events (TDEs), with peak magnitudes of $g \sim 20.0$ and $g \sim 19.6$, respectively. Blackbody fits to the multi-band photometry yield roughly constant temperatures of $\sim 10^{4}$ K for both sources, with peak bolometric luminosities of $10^{43.6}$ erg s$^{-1}$ and $10^{43.8}$ erg s$^{-1}$, respectively. Optical spectra obtained during the declining phase display characteristic broad Balmer, He II, and N III emission lines, classifying both sources as H+He TDEs. Host-galaxy SED fitting suggests stellar masses on the order of $10^{9} M_{\odot}$ and black hole masses of $\sim10^{6} M_{\odot}$, which are typical of optically selected TDE hosts. The pilot discovery of these two faint TDEs along with dozens of candidates, demonstrates the capability of WFST to discover TDEs, owing to its exceptional survey depth. Furthermore, these events highlight the value of the $u$ band in multi-band surveys for TDE selection, particularly for distinguishing TDEs from supernovae at early phases. With continued operations, WFST is expected to substantially expand the known TDE population, especially by uncovering nearby low-luminosity events and high-redshift TDEs, both of which remain poorly explored.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.24203-b31b1b.svg)](https://arxiv.org/abs/2611.24203) | **WFST0533buha (Bi'an): A Fast Blue Ultraluminous Transient Candidate with Unusual Blue/NUV Colors**  |
|| Z. Xu, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *15 pages, 11 figures, 4 tables. Revised manuscript submitted to AAS Journals*|
|**Abstract**|            We present WFST0533buha (Bi'an), a fast blue ultraluminous transient (FBUT) candidate discovered by the Wide Field Survey Telescope Deep High-cadence $ugr$-band Survey. The source brightened by approximately 2.5 mag in $u$ within 3 days and has a half-maximum timescale ($t_{1/2}$) shorter than 6 days. Two spectra show a blue, featureless continuum but do not provide a secure redshift. Einstein Probe observations yield only X-ray upper limits, while a faint radio counterpart was detected at 15 GHz with the VLA. The early $u$-band detection and intensive multiband coverage of Bi'an demonstrate the value of deep, high-cadence blue-band observations for discovering and characterizing FBUTs at early phases. Notably, Bi'an shows redder blue/NUV colors than AT 2018cow and AT 2024wpp, suggesting possible diversity in the short-wavelength properties of FBUTs.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.24204-b31b1b.svg)](https://arxiv.org/abs/2611.24204) | **SN 2025wiu: A Slowly Evolving Type Ib Supernova with an Extraordinary Post-maximum Bump**  |
|| Z. Liu, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *20 pages, 9 figures, 5 tables. Revised manuscript submitted to AAS Journals*|
|**Abstract**|            Stripped-envelope supernovae (SESNe) originate from massive progenitors that have lost most or all of their outer envelopes prior to core collapse. Recent advances in optical time-domain surveys have revealed a growing population of peculiar SESNe with unusual light-curve morphologies, pointing to energy sources beyond the standard radioactive-decay scenario. In this work, we present multi-band photometric and spectroscopic observations of SN 2025wiu, a peculiar Type Ib supernova discovered by the Wide Field Survey Telescope (WFST). SN 2025wiu is characterized by a high luminosity and exceptionally slow photometric evolution, with a prominent post-maximum bump peaking at $\sim50$ days after the primary maximum. Its early light curve also shows a weak excess consistent with shock-cooling emission from a compact progenitor with a radius of $\sim4\,R_\odot$. The spectra show only modest evolution, with persistently weak line features and no narrow emission lines detected even at $\sim110$ days after maximum. By modeling the primary peak and late-time tail with a combination of radioactive decay and additional energy sources, we isolate the post-maximum bump, which has a roughly symmetric profile with a peak luminosity of $\sim2.4 \times 10^{42}~\mathrm{erg\,s^{-1}}$ and a characteristic duration of $\sim42$ days. We discuss circumstellar medium (CSM) interaction, accretion, magnetar activity, and a double $^{56}$Ni distribution as possible origins of the bump. The latter three scenarios face challenges in reproducing the post-maximum bump, while delayed CSM interaction remains a possible explanation.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.24387-b31b1b.svg)](https://arxiv.org/abs/2611.24387) | **Radio emission from a local Little Red Dot: a challenge to the ionized cocoon scenario**  |
|| G. Mazzolari, et al. -- incl., <mark>X. Lin</mark> |
|*Appeared on*| *2026-11-19*|
|*Comments*| *8 pages (+3 pages in Appendix), 2 main Figures. Submitted to ApJL*|
|**Abstract**|            Several scenarios have been proposed to explain the properties of Little Red Dots (LRDs), invoking either heavy obscuration by dense neutral gas, super-Eddington accretion, or an ionized gas cocoon that produces electron scattering. Radio observations can distinguish between these scenarios. Radio emission is largely unaffected by neutral gas and dust obscuration, whereas free-free absorption can suppress it in the presence of large free electron columns. Here, we investigate the origin of radio emission from the local LRD J1047. We jointly analyze its radio data and the broad H$\alpha$ emission from optical spectroscopy to test the predictions of different physical scenarios. We find that the observed radio luminosity must have an AGN origin, and it is consistent with that of a standard radio-quiet AGN obscured by dense neutral gas. Pure super-Eddington accretion underpredicts the radio luminosity by a factor of $\sim 30$, although a combination of super-Eddington accretion and obscuration remains viable. Interestingly, the statistically preferred model for the H$\alpha$ profile is an exponential fit, which is often associated with electron scattering in LRDs. This interpretation implies an electron column density of $N_e=2.4\times10^{24}\rm~cm^{-2}$ and a strong free-free absorption ($\tau_{\rm ff,5GHz}\sim 10^6$), incompatible with the radio detection. This tension suggests that the exponential profiles of J1047 more likely trace a stratified broad line region rather than electron scattering, with potential implications for virial black-hole mass estimates also in high-$z$ LRDs. Less likely, the radio emission may escape through a partially covering ionized medium or originate outside the putative scattering cocoon.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.24561-b31b1b.svg)](https://arxiv.org/abs/2611.24561) | **Gaia Science Alerts microlensing events with mass estimates verifiable in Gaia Data Release 4**  |
|| M. Wicker, et al. |
|*Appeared on*| *2026-11-19*|
|*Comments*| *17 pages, 12 figures. Accepted for publication in Sect. 11. Celestial mechanics and astrometry of Astronomy & Astrophysics on 16 October 2026*|
|**Abstract**|            Gaia Science Alerts provide an all-sky stream of photometric transients, including events consistent with gravitational microlensing. For such events, forthcoming epoch astrometry from the Gaia mission offers the possibility of testing photometric lens mass predictions. While microlensing parallax inferred from the photometric light curve can constrain the event geometry, a direct lens mass measurement additionally requires the angular Einstein radius, which may be accessible through astrometric microlensing. We aim to construct a pre-Data-Release-4 (DR4) sample of Gaia microlensing events for which the available photometry provides a measurable microlensing-parallax signal. This allows us to derive statistical predictions for the lens mass and distance. We forward-modelled these events to predict the detectability of the astrometric microlensing signal in Gaia DR4. We identify 56 Gaia Science Alerts microlensing events with 71 point-source point-lens solutions with annual parallaxes that satisfy our fit-quality, posterior-boundedness, and parallax-significance criteria. Lens masses and distances were inferred using two prior-based frameworks, pyLIMASS and the Besançon Galaxy Model, which combine the measured microlensing parameters and Gaia source information with stellar-population and Galactic models. These inferred parameters were used to simulate Gaia DR4-like along-scan astrometry and estimate the detectability of the astrometric microlensing signal. The inferred lens masses and distances remain strongly prior-dependent without a direct angular Einstein radius measurement. Among the 284 event-solution-prior branches, 47 are classified as high-confidence astrometric detections using Gaia DR4-like simulations. These predictions can be tested once epoch astrometry is released in Gaia DR4.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Max' keyword not found.</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.23367/./ngc253_ratio_Draine_BC03_Z0.02_10Myr_by_streamer_U_F3p3_F11p3_disksub_noSE.png', 'tmp_2611.23367/./streamer_zooms.png', 'tmp_2611.23367/./ngc253_streamer_emission_profiles_disksub_byStreamer_noSE.png']
copying  tmp_2611.23367/./ngc253_ratio_Draine_BC03_Z0.02_10Myr_by_streamer_U_F3p3_F11p3_disksub_noSE.png to _build/html/
copying  tmp_2611.23367/./streamer_zooms.png to _build/html/
copying  tmp_2611.23367/./ngc253_streamer_emission_profiles_disksub_byStreamer_noSE.png to _build/html/
exported in  _build/html/2611.23367.md
    + _build/html/tmp_2611.23367/./ngc253_ratio_Draine_BC03_Z0.02_10Myr_by_streamer_U_F3p3_F11p3_disksub_noSE.png
    + _build/html/tmp_2611.23367/./streamer_zooms.png
    + _build/html/tmp_2611.23367/./ngc253_streamer_emission_profiles_disksub_byStreamer_noSE.png
found figures []
exported in  _build/html/2611.23492.md
found figures ['tmp_2611.24630/./figures/MZR_N2obs-sim.png', 'tmp_2611.24630/./figures/SFMS_residuals.png', 'tmp_2611.2

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand\eleven{11.3~\micron}$
$\newcommand\seven{7.7~\micron}$
$\newcommand\six{6.2~\micron}$
$\newcommand\three{3.3~\micron}$
$\newcommand\ElevenSeven{11.3/7.7~\micron}$
$\newcommand\ThreeSeven{3.3/7.7~\micron}$
$\newcommand\ThreeEleven{3.3/11.3~\micron}$
$\newcommand\kms{km~s^{-1}}$
$\newcommand\ha{H\alpha}$
$\newcommand\LTIR{L_{\rm TIR}}$
$\newcommand\LPAH{L_{\rm PAH}}$
$\newcommand\qpah{q_{\rm PAH}}$
$\newcommand\Msun{M_{\odot}}$
$\newcommand\Msunperyear{M_{\odot}~yr^{-1}}$
$\newcommand\LHands{L.~Hands et al.~(in prep)}$
$\newcommand\FDonnan{F.~Donnan et al.~submitted}$</div>



<div id="title">

# JWST Observations of Starbursts: Polycyclic Aromatic Hydrocarbons in the NGC 253 Superwind

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.23367-b31b1b.svg)](https://arxiv.org/abs/2611.23367)<mark>Appeared on: 2026-11-19</mark> -  _18 pages, 5 main figures, 4 appendix figures, submitted to ApJL_

</div>
<div id="authors">

S. A. Cronin, et al. -- incl., <mark>J. Smith</mark>, <mark>F. Walter</mark>

</div>
<div id="abstract">

**Abstract:** We present the first spatially resolved study of polycyclic aromatic hydrocarbon (PAH) evolution in the galactic superwind of the nearby ( $d \sim 3.5$ Mpc) galaxy NGC 253. Using JWST MIRI and NIRCam, we image $\sim \pm2$ kpc along the galaxy minor axis, capturing both sides of the starburst-driven wind at $\sim0\farcs05{-}0\farcs375$ ( $\sim0.85{-}6.4$ pc) resolution and providing one of the most detailed views of the wind to date. We target PAH emission at $\three$ , $\seven$ , and $\eleven$ , revealing PAH-bright filamentary streamers along the limb-brightened edges of the outflow cones out to at least a deprojected distance $r \sim 0.5$ kpc, beyond which they likely become confused with the spiral arms. In the southwest (SW) and northeast (NE) streamers, PAH intensity declines faster than $r^{-2}$ , suggesting either PAH destruction or a decrease in gas column density or dust-to-gas ratio rather than a response to declining illumination alone. Band ratios trace changes in PAH properties, with the $\ElevenSeven$ ratio increasing with distance in the SW and northwest (NW) streamers, and $\ThreeSeven$ and $\ThreeEleven$ decreasing in the SW and NE streamers, indicating larger, more neutral PAHs farther from the starburst. Given the thermal and dynamical properties of the gas in the SW streamer, cool gas is expected to survive acceleration in the outflow and enable PAH shielding. Compared with those of the prototypical starburst M82, the NGC 253 band ratios indicate faster evolution in PAH size. Thus, the NGC 253 wind may represent an earlier stage of outflow evolution in which small PAHs are destroyed near the starburst within the first few Myr.

</div>

<div id="div_fig1">

<img src="tmp_2611.23367/./ngc253_ratio_Draine_BC03_Z0.02_10Myr_by_streamer_U_F3p3_F11p3_disksub_noSE.png" alt="Fig5" width="100%"/>

**Figure 5. -** Ratios from Figure \ref{fig:ratio_profiles} compared with continuum-subtracted [Draine, et. al (2021)](https://ui.adsabs.harvard.edu/abs/2021ApJ...917....3D) dust models, assuming a $3{-}10$ Myr starburst at solar metallicity  ([Bruzual and Charlot 2003](https://ui.adsabs.harvard.edu/abs/2003MNRAS.344.1000B)) . Each grid corresponds to a radiation field intensity $U$ at $A_{\rm V} = 0$. The small black arrow shows the magnitude and direction of how the grids shift for a higher extinction of $A_{\rm V} = 5$, and the long black arrows show the effects of PAH size and ionization. Points are colored by distance from the nucleus. The gray arrow shows the M82 trend from $0.3$ kpc (excluding inner points affected by silicate absorption) to $0.6$ kpc (the most extended NGC 253 streamer). White symbols show median PHANGS-JWST ratios across galaxies  ([Pathak, et. al 2026](https://ui.adsabs.harvard.edu/abs/2026ApJ..1008...35P))  for star-forming disks, disk-like galaxy centers ("disky center"), and young star-forming centers or central molecular zones (CMZs). Silicate absorption affecting the \eleven feature near the starburst may explain some ratios outside the $A_{\rm V} = 0$ grids. In the SW streamer, PAHs become larger and more neutral with distance, matching M82 but spanning a wider range over the same distance and timescale. NGC 253 ratios fall near the PHANGS-JWST medians, with the outermost SW and NE ratios matching disks and disk-like centers, and the innermost ratios matching CMZs. Figure \ref{fig:draine_models_no_disksub} shows the NGC 253 ratios without disk subtraction. (*fig:draine_models*)

</div>
<div id="div_fig2">

<img src="tmp_2611.23367/./streamer_zooms.png" alt="Fig2" width="100%"/>

**Figure 2. -** Emission maps of the \three, \seven, and \eleven PAH features in the central $2\times2$ kpc region of NGC 253. The \three map is continuum-subtracted from F335M using two flanking filters (F250M and F360M), while the \seven and \eleven maps are both background- and continuum-subtracted. The continuum images at all three wavelengths are in Figure \ref{fig:continuum} of the Appendix. Labeled dashed rectangles mark the four wind streamers where we extract measurements (zoom-ins in the bottom row to show detailed sub-structure). Unlabeled dotted rectangles mark the adjacent off-streamer regions used to estimate disk-only PAH emission, since the disk contributes to each line of sight. (*fig:emission_maps*)

</div>
<div id="div_fig3">

<img src="tmp_2611.23367/./ngc253_streamer_emission_profiles_disksub_byStreamer_noSE.png" alt="Fig3" width="100%"/>

**Figure 3. -** Binned-mean emission profiles extracted along each NGC 253 streamer (colored points) with M82 (gray points) and arbitrary $r^{-2}$ and $r^{-3}$ profiles for comparison. The shaded regions are the standard error of the mean.
    Each profile is multiplied by $r^{2}$ to divide out the geometric dilution expected for illumination by the starburst, so that a flat ($r^{-2}$) profile indicates changes in PAH emission due to distance from the starburst. The top $x{-}$axis converts the deprojected distance to a timescale assuming a cool-outflow speed of $200$ \kms. All PAH bands in the SW and NE streamers decline faster than $r^{-2}$, suggesting possible changes to PAH emission that are not due to distance from the illuminating starburst. The NW streamer roughly follows $\sim r^{-2}$ much like M82. We exclude the SE streamer, where we do not produce a reliable disk subtraction. (*fig:emission_profiles*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.23367"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$</div>



<div id="title">

# JEPA for astronomy: \ multi-survey spectral alignment puts stellar parameters on the leading principal axes

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.23492-b31b1b.svg)](https://arxiv.org/abs/2611.23492)<mark>Appeared on: 2026-11-19</mark> -  _15 pages, 5 figures, 2 tables. Submitted to AAS journals. Comments welcome_

</div>
<div id="authors">

X. Zhao, R. F. Wyse, <mark>Y.-S. Ting</mark>, A. S. Szalay

</div>
<div id="abstract">

**Abstract:** Spectra from different surveys carry different systematics and are not calibrated against one another. Multi-survey foundation models for stellar spectra are then built by aligning embeddings of the same star observed across surveys. Most in astronomy follow the recipes established for language and vision, aligning with a contrastive loss that separates every star from the others. However, this does not ensure that the stellar physics is encoded in a form that a simple read-out can recover. Holding the cross-survey architecture fixed and varying only the alignment loss, we measure what an objective encodes from the principal-component geometry of the Gaia BP/RP (XP) spectra and from a linear probe of the stellar parameters. In a properly aligned embedding, the leading two principal components form a plane that recovers an analog of the Kiel diagram, the third component tracks [ Fe/H ] , and [ $\alpha$ /Fe ] varies coherently across the plane, such that the chemically defined Galactic disks and accreted populations occupy distinct loci. We show that legibility, the ease of reading the information out, is highest for a training objective that never pushes stars apart, and instead constrains only the overall distribution of each survey's embeddings (JEPA). JEPA also exceeds the contrastive alternative at the same architecture, and the performance improves further as more surveys (Gaia XP $+$ LAMOST LRS $+$ DESI $+$ APOGEE) are included. Predicted abundances for 8.0 million XP giants reproduce the main [ $\alpha$ /Fe ] – [ Fe/H ] structure of the Milky Way. These results indicate that the choice of alignment objective, not alignment alone, determines whether stellar labels are linearly recoverable.

</div>

<div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.23492"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\ha}{H\alpha}$
$\newcommand{\niir}{[\ion{N}{ii}] \lambda6584}$
$\newcommand{\niib}{[\ion{N}{ii}] \lambda6549}$
$\newcommand{\niiha}{[\ion{N}{ii}] \lambda6584/ \rm H\alpha}$
$\newcommand{\siiha}{[\ion{S}{ii}] \lambda\lambda6716,6731/ \rm H\alpha}$
$\newcommand{\siir}{[\ion{S}{ii}] \lambda6731}$
$\newcommand{\siib}{[\ion{S}{ii}] \lambda6716}$
$\newcommand{\labelitemi}{\bullet}$
$\newcommand{\arraystretch}{1.5}$
$\newcommand{\arraystretch}{1.5}$</div>



<div id="title">

# The Gas-Phase Metallicity and Nitrogen abundances in typical galaxies at $z\sim5$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.24630-b31b1b.svg)](https://arxiv.org/abs/2611.24630)<mark>Appeared on: 2026-11-19</mark> -  _Main text 16 pages, 8 figures. Main results in Fig. 5 and Fig. 6. Submitted to A&A. Comments are welcome_

</div>
<div id="authors">

C. D. Cesare, et al. -- incl., <mark>J. Matharu</mark>

</div>
<div id="abstract">

**Abstract:** The relation between stellar mass and gas-phase metallicity (MZR) is a key tracer of the baryon cycle in galaxies. JWST now gives access to rest-optical emission lines at $z > 3$ , enabling unprecedented constraints on the chemical enrichment and the conditions of the interstellar medium of galaxies in the early Universe. In this work, we present the MZR at $z=3.8-6.6$ from stacked NIRCam/grism spectra of 1748 H $\alpha$ -selected star-forming galaxies compiled over an area of 320 arcmin $^2$ . The simple H $\alpha$ flux-limited selection allows us to measure representative $[\ion{N}{ii}] \lambda6584/\rm H\alpha$ and $[\ion{S}{ii}] \lambda\lambda6716,6731/\rm H\alpha$ ratios in stacks spanning $\rm log(M_\star/M_\odot)=8-10.2$ , extending the upper mass range of previous high-redshift MZR measurements by $\sim1$ dex. Gas-phase metallicities are derived from these ratios using strong-line calibrations recently derived from high-redshift galaxies. The [ $\ion{N}{ii}$ ] -based MZR has a slope $\gamma = 0.42^{+0.10}_{-0.07}$ and $12+\rm log(O/H) = 7.94^{+0.06}_{-0.09}$ at $\rm log(M_\star/M_\odot) = 9$ , in broad agreement with recent empirical studies and hydrodynamical simulations. The [ $\ion{S}{ii}$ ] -based relation yields a consistent slope and a $\sim0.3$ dex higher normalization, within the systematic uncertainties of strong-line calibrations. From the resolved $\rm [\ion{S}{ii}]$ doublet we infer a  median electron density of $n_e = 184^{+226}_{-169} \rm cm^{-3}$ . Combining $\rm [\ion{N}{ii}]$ and $\rm [\ion{S}{ii}]$ , we indirectly estimate the N/O ratio and find that it follows the trend expected from primary and secondary nitrogen production, with no evidence for nitrogen enhancement in the H ii region emission that dominates our galaxies. Our results provide a benchmark for the chemical enrichment and ISM conditions of typical star-forming galaxies at $z\sim5$ across a wide mass range.

</div>

<div id="div_fig1">

<img src="tmp_2611.24630/./figures/MZR_N2obs-sim.png" alt="Fig8" width="100%"/>

**Figure 8. -** The mass-metallicity relation inferred from our measurements, comparison with observations and simulations. Blue hexagons show the metallicity estimates derived from the stacked galaxy sample in each mass bin using the N2-based diagnostic, the uncertainties account for measurement uncertainties and calibration systematics reported in [Sanders, Shapley and Topping (2026)](). The open symbols corresponds to the gas-phase metallicity based on the $\niir$  marginal detection with SNR$\sim 2$. **Left:** Together with our estimates, we report metallicity measurements for individual galaxies obtained using JWST/NIRSpec (gray open symbols). In particular, we show data from [Chemerynska, Atek and Dayal (2024)]()(triangles), [Chakraborty, Sarkar and Smith (2025)]()(squares), [Stanton, Cullen and Carnall (2026)]()(diamonds) and [Faisst, Liu and Dubois (2026)]()(circles). We also show results from stacked NIRCam/grism \citep[][pink]{Kotiwale+2026} and NIRSpec spectra \citep[][orange]{Lam+2026}, and average metallicity estimates  ([Curti, Maiolino and Curtis-Lake 2024](), [Nakajima, Ouchi and Xu 2022]())  obtained from NIRSpec spectroscopy (orange). Yellow squares show the stacked measurements at $z\sim2.3$ from [Sanders, Shapley and Jones (2021)](). **Right:** Comparison between our metallicity estimates and predictions from hydrodynamical simulations. Dashed lines show median values - with 16th and 84th percentiles - for SPHINX$^{20}$ ([Rosdahl, Katz and Blaizot 2018](), [Katz, Rosdahl and Kimm 2023]())  at $z = 4.64-6$, FLARES  ([Lovell, Vijayan and Thomas 2021](), [Vijayan, Lovell and Wilkins 2021]())  at $z = 5-7$, COLIBRE  ([Schaye, Chaikin and Schaller 2026](), [Sharda, Schaye and McGibbon 2026]())  at $z = 5$ and \texttt{dustyGadget} ([Graziani, Schneider and Ginolfi 2020](), [Graziani and Schneider 2023](), [Schneider and Graziani 2026]())  at $z=4.5$. Solid lines show the best-fit results from THESAN-ZOOM   ([Kannan, Puchwein and Smith 2025](), [McClymont, Tacchella and Smith 2026]()) , FIRE-2  ([Ma, Hopkins and Garrison-Kimmel 2018](), [Marszewski, et. al 2024]())  and FirstLight  ([Ceverino, Glover and Klessen 2017](), [Langan, Ceverino and Finlator 2020]())  at $z = 5$. (*fig:mzr*)

</div>
<div id="div_fig2">

<img src="tmp_2611.24630/./figures/SFMS_residuals.png" alt="Fig1" width="100%"/>

**Figure 1. -** The star-forming main sequence at $z\sim5$. We show our sample (gray), together with median values and literature relations \citep[][]{Speagle+2014, Clarke+2025,DiCesare+2026}. Dotted lines indicate extrapolations of the literature relations below their respective lower stellar-mass limit. The vertical dashed line marks the stellar-mass completeness limit at $\rm log(M_\star/M_\odot) > 8.5$. Individual galaxies with $\niir$  detected at SNR $>5$ are highlighted in blue, while those with $\siib$  or $\siir$  detected at SNR $> 5$ are highlighted in orange. The histogram shows the number of galaxies in each stellar-mass bin used for stacking (see Sect. \ref{sec:mass_bins}). The bottom panel shows the offset in SFR from the SFMS of [Clarke, Shapley and Lam (2025)](); the dotted line indicates where the relation is extrapolated and shaded band shows the intrinsic scatter. Filled circles show the median offset of each survey in the same stellar-mass bins as our stacks and open circles show our stacked measurements. The pink line shows the relation of [Matthee and Naidu (2026)]() relative to that of [Clarke, Shapley and Lam (2025)](). (*fig:sfms*)

</div>
<div id="div_fig3">

<img src="tmp_2611.24630/./figures/N2logNO-OH.png" alt="Fig2" width="100%"/>

**Figure 2. -** The nitrogen-to-oxygen (N/O) abundance ratio as a function of gas-phase metallicity estimated based on N2-diagnostic. Blue hexagons show the results from our stacked sample in each mass bin. The blue open symbol shows the gas-phase metallicity based on the $\niir$  marginal detection with SNR$\sim 2$. For comparison, we show local Hii regions from [Izotov, et. al (2006)](), [Berg, Pogge and Skillman (2020)]()(gray), together with the local N/O-O/H relations of [Nicholls, et. al (2017)]()(gray dashed line) and [Scholte, Cullen and Moustakas (2026)]()(red solid line).
   We also include measurements of individual high-$z$ galaxies  ([Arellano-Córdova, Cullen and Carnall 2025](), [Cameron, Carreira and Simmonds 2026](), [Schaerer, Izotov and Marques-Chaves 2026]()) , stacked sample at $z\sim6$ ([Umeda, Ouchi and Nakajima 2026]())  and median values at $z\sim 1-6$ ([Cataldi, Belfiore and Curti 2026]()) . Large and small diamonds indicate measurements from [Cameron, Carreira and Simmonds (2026)](), based on direct-$T_e$ and strong-line metallicity calibrations, respectively. All high-$z$ N/O estimates are based on detections of the optical $[$\ion${N}{II}]$ line. The orange Sun symbol marks the solar abundance values from [Asplund, Amarsi and Grevesse (2021)](). (*fig:no-oh*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.24630"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

156  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

23  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

7  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
